# 03 · Data Cleaning
**Goal:** Fix all data-quality problems (including what `02_data_modeling.ipynb` has found), so that `04_data_integration.ipynb` can merge the four tables without data errors.

## Open decisions (not yet made — decide here, then implement)

- [ ] **Menu dedup rule.** Menu has 242 rows but only 106 unique `(product_name, size)` keys, and 24 of those groups have genuinely different `calories` / `price_usd` / `sugar_g` / `caffeine_mg` values (not harmless duplicate rows — see `02_data_modeling.ipynb`, the Caffè Latte Grande example ranges \$3.99–\$4.49 and 70–240 calories). Need to pick an aggregation rule to collapse each group to one row (e.g. median — more robust to outliers than mean given the spread) before Menu can be used as a dimension table.
- [ ] **2026-03 Macro gap.** Macro data ends `2026-02-01`; 122 transactions fall in `2026-03` with no matching macro month. Decide: leave `cpi`/`avg_hourly_earnings`/`real_wage_index` NULL for those rows, forward-fill the last known macro values, or drop the 122 rows.
- [ ] **Master dataset enrichment.** Decide whether to pull `weather.temp_max_f` / `weather.temp_min_f` and/or `macro.avg_hourly_earnings` / `macro.real_wage_index` into the master dataset (currently only `temp_f` and `cpi` are embedded in Transaction). This affects the join in `04_data_integration.ipynb`, so decide it here before writing that notebook's join code.

### Nulls that need to resolve
Discovered from `01_data_audit.ipynb`'s `.isnull().sum()` output:

- [x] `transactions.cpi` / `transactions.total_price` — 4,324 nulls each. Rows dropped (see Step 3.1).
- [x] `transactions.caffeine_mg` — 8,186 nulls. Not recoverable in-table or from `menu` right now; left NULL, deferred to after Menu dedup (see Step 3.2).
- [x] `menu.caffeine_mg` — 23 nulls. Deferred — resolves as a side effect of Menu dedup in Step 4 (see Step 3.3).
- [x] `macro.cpi` / `macro.real_wage_index` — 1 null each. Fixed via linear interpolation (see Step 3.4).

### 1. Load Data

In [17]:
import pandas as pd

menu         = pd.read_csv('../data/raw/starbucks_menu.csv')
transactions = pd.read_csv('../data/raw/synthetic_transactions.csv')
macro        = pd.read_csv('../data/raw/fred_macro.csv')
weather      = pd.read_csv('../data/raw/weather_daily.csv')

### 2. Data Overview

Most data overview are already covered in `01_data_audit.ipynb`.
Additional step of checking duplicates is added below:

In [18]:
menu.duplicated().sum()

np.int64(1)

In [19]:
transactions.duplicated().sum()

np.int64(0)

In [20]:
macro.duplicated().sum()

np.int64(0)

In [21]:
weather.duplicated().sum()

np.int64(0)

### 3. Cleaning Missing Values
Assess missing values in each dataset; determine whether they need removal, imputation, or retained.

#### 3.1 `transactions.cpi` and `transactions.total_price`

Since `transactions.cpi` has 4,324 nulls and `transactions.total_price` also has 4,324 nulls, I will check whether they are the same rows.

In [22]:
transactions[transactions['cpi'].isnull() & transactions['total_price'].isnull()]

,transaction_id,date,hour,time_slot,city,persona,is_weekend,temp_f,cpi,category,product_name,size,base_price,customizations,n_customizations,upcharge,total_price,calories,sugar_g,caffeine_mg
30,TXN-000031,2025-09-30,11,late_morning,Chicago,student,0,70.3,NaN,Frappuccino® Blended Coffee,Coffee,Venti,5.43,none,0,0.0,NaN,310,69,95.0
48,TXN-000049,2025-09-06,7,morning_rush,New York,health_conscious,1,74.6,NaN,Coffee,Brewed Coffee,Grande,2.55,none,0,0.0,NaN,5,0,330.0
54,TXN-000055,2025-09-30,12,lunch,Houston,weekend_explorer,0,82.4,NaN,Frappuccino® Light Blended Coffee,Java Chip,Grande,4.59,none,0,0.0,NaN,220,39,105.0
93,TXN-000094,2025-09-06,12,lunch,New York,afternoon_treat,1,74.6,NaN,Frappuccino® Blended Coffee,Coffee,Grande,4.79,Vanilla Syrup|Oat Milk,2,1.3,NaN,180,36,70.0
157,TXN-000158,2025-09-18,11,late_morning,Los Angeles,weekend_explorer,0,73.9,NaN,Coffee,Brewed Coffee,Tall,2.18,none,0,0.0,NaN,4,0,260.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99913,TXN-099914,2025-09-11,8,morning_rush,Houston,health_conscious,0,82.1,NaN,Classic Espresso Drinks,Caffè Mocha (Without Whipped Cream),Tall,3.61,Caramel Drizzle,1,0.6,NaN,170,27,95.0
99928,TXN-099929,2025-09-02,15,afternoon,New York,weekend_explorer,0,69.9,NaN,Frappuccino® Light Blended Coffee,Mocha,Grande,4.47,Mocha Sauce,1,0.6,NaN,150,30,95.0
99952,TXN-099953,2025-09-09,16,afternoon,Houston,afternoon_treat,0,75.8,NaN,Signature Espresso Drinks,Caramel Macchiato,Venti,5.90,Oat Milk,1,0.7,NaN,240,41,150.0
99969,TXN-099970,2025-09-28,9,morning_rush,Chicago,student,1,70.1,NaN,Frappuccino® Blended Coffee,Coffee,Grande,4.79,Caramel Drizzle,1,0.6,NaN,180,36,70.0


Exactly 4324. So they are the same rows. This also means that the missing values of `cpi` and `total_price` might because of a same problem.

In [23]:
missing_rows = transactions[(transactions['cpi'].isnull()) & (transactions['total_price'].isnull()) ]

missing_rows['date'].unique()

array(['2025-09-30', '2025-09-06', '2025-09-18', '2025-09-19',
       '2025-09-10', '2025-09-07', '2025-09-08', '2025-09-23',
       '2025-09-28', '2025-09-25', '2025-09-29', '2025-09-04',
       '2025-09-12', '2025-09-09', '2025-09-03', '2025-09-13',
       '2025-09-17', '2025-09-27', '2025-09-24', '2025-09-15',
       '2025-09-22', '2025-09-02', '2025-10-01', '2025-09-11',
       '2025-09-05', '2025-09-20', '2025-09-16', '2025-09-21',
       '2025-09-14', '2025-09-26'], dtype=object)

The dates of those rows are almost entirely September 2025, plus one day in October. That's not the 2026-03 macro gap. check whether `Macro` itself is missing September 2025 data.

In [24]:
macro[macro['date'].str.startswith('2025-09')]

,date,cpi,avg_hourly_earnings,real_wage_index
54,2025-09-01,324.245,36.7,11.3186


`Macro` has a real September 2025 CPI value, so this isn't a macro coverage gap. The nulls in `Transactions` look like corrupted in the raw file itself.

The `cpi` value in those rows could be recovered by `cpi` in `Macro`, but `total_price` should also be recovered at the same time to make those rows be able to use.

Next step:
check whether `total_price` is recoverable.

`base_price` and `upcharge` are intact on every one of these rows, so check whether `total_price` is a fixed function of `(product_name, size, upcharge)`. If it is, the correct value could be looked up from other rows with the same combination:

In [25]:
transactions.groupby(['product_name', 'size', 'upcharge'])['total_price'].nunique().value_counts()

total_price
1     392
2     217
3     143
5     100
18     99
19     93
4      93
6      82
17     72
7      68
9      66
12     64
8      63
15     56
13     55
11     54
10     51
16     42
14     29
20     14
0      11
30      3
25      3
21      2
27      2
31      1
22      1
36      1
33      1
24      1
23      1
Name: count, dtype: int64

Most `(product_name, size, upcharge)` combinations have far more than one distinct `total_price`. Which means that pricing depends on factors beyond these three, so there's no reliable way to reconstruct the true value.

**Decision: drop all these 4,324 rows.** They're a small share of the dataset (~4.3%), but mostly concentrated in September 2025. Flag that in Step 9 to notify that that month's transaction count may be understated.

In [26]:
transactions = transactions[~(transactions['cpi'].isnull() & transactions['total_price'].isnull())].reset_index(drop=True)
#delete rows with NULL total_price from transactions.

transactions.shape

(95676, 20)

Finished cleaning nulls of `transactions.cpi` and `transactions.total_price`

#### 3.2 `transactions.caffeine_mg`

8,186 nulls remain. Check which products they belong to:

In [ ]:
transactions[transactions['caffeine_mg'].isnull()]['product_name'].value_counts()

Only 6 products, but check whether `caffeine_mg` is a fixed value per `(product_name, size)` — same logic as the `total_price` check above:

In [ ]:
transactions.groupby(['product_name', 'size'])['caffeine_mg'].nunique().value_counts()

Most `(product_name, size)` groups have exactly 1 known value (good, but they already have no nulls, so there's nothing to fill there). A handful have 0 — meaning every single row for that product/size is null, so there's no value anywhere in `transactions` to copy from. (A separate few groups — `Coffee` — have 2 different values with no nulls at all; that's a consistency problem for Step 6, not a missing-value problem.)

Check whether `menu` has the value for the products that are null everywhere in `transactions`:

In [ ]:
null_products = transactions[transactions['caffeine_mg'].isnull()]['product_name'].unique()
menu[menu['product_name'].isin(null_products)][['product_name', 'size', 'caffeine_mg']].sort_values(['product_name', 'size'])

`menu` isn't a reliable source for these either: some of these products are entirely null in `menu` too (e.g. `Tazo® Tea`), and the rest sit inside the messy duplicate-key groups already flagged in the Menu dedup open decision — pulling a value from a group that has conflicting rows would just be guessing which duplicate to trust.

**Decision: leave `transactions.caffeine_mg` nulls as NULL for now.** It's a secondary nutritional field, not required for revenue/portfolio analysis, so it's low-risk to leave empty. Revisit after the Menu dedup decision is implemented (Step 4) — once each `(product_name, size)` in `menu` collapses to one trustworthy value, come back and re-check whether any of these can be filled from it.

#### 3.3 `menu.caffeine_mg`

These 23 nulls sit inside the same duplicate-key groups covered by the Menu dedup open decision above. Once that dedup runs (e.g. taking the median per `(product_name, size)`), the aggregation will skip NaN automatically — so any group that has at least one real value will end up with a clean, non-null result for free. No separate fix needed here; resolve this as a side effect of Step 4.

#### 3.4 `macro.cpi` / `macro.real_wage_index`

In [ ]:
macro[macro['cpi'].isnull() | macro['real_wage_index'].isnull()]

Both nulls are on the same row (`2025-10-01`) — `avg_hourly_earnings` is present there, only `cpi` and `real_wage_index` are missing. These are smooth, slow-moving monthly indicators (`cpi` went `324.245 → NaN → 325.063` from Sep to Nov), so linear interpolation between the neighboring months is a reasonable, low-risk fix — much safer than for `total_price`, since macro data doesn't have the sharp jumps individual transactions can have.

`.interpolate()` fills each NaN by drawing a straight line between the nearest valid values before and after it (needs the rows sorted by date, which `macro` already is):

In [ ]:
macro[['cpi', 'real_wage_index']] = macro[['cpi', 'real_wage_index']].interpolate()
macro.isnull().sum()

### 4. Cleaning Duplicate Records

### 5. Data Type Conversion

### 6. Text Standardization

### 7. Cleaning Invalid / Impossible Values

### 8. Outlier

### 9. Cleaning Summary & Cleaned Dataset